# 02 — Chronological forecasting experiments

This notebook follows `01_Data_Exploration.ipynb` and leaves it unchanged. It explains the 28-day validation design, summarizes the 5,000-series pilot, and reads the generated CSVs so results can be inspected after each run.

**Research question:** How effectively does an AI-based model forecast retail demand under volatile demand conditions, compared with traditional baselines?

The current pilot uses 5,000 seeded item-store series, three non-overlapping 28-day holdouts, and 1,000 paired series-cluster bootstrap replicates. The groups are fold-specific terciles of CV across positive-demand weekly totals in the preceding 52 complete weeks.

## Pilot results

Pooled scores combine absolute error and demand across all three folds. Positive WAPE improvement means the baseline has higher error than LightGBM. Confidence intervals resample series IDs and retain all fold observations for each selected series.

| Volatility group | Baseline | Baseline WAPE | LightGBM WAPE | Improvement (WAPE points) | 95% bootstrap interval | Relative improvement |
|---|---|---:|---:|---:|---:|---:|
| All | Four-week weekday mean | 0.7764 | 0.6928 | 0.0836 | 0.0758–0.0919 | 10.77% |
| Low | Four-week weekday mean | 0.6206 | 0.5808 | 0.0398 | 0.0325–0.0482 | 6.42% |
| Moderate | Four-week weekday mean | 0.9549 | 0.8333 | 0.1216 | 0.1100–0.1330 | 12.74% |
| High | Four-week weekday mean | 1.0272 | 0.8602 | 0.1671 | 0.1470–0.1886 | 16.26% |

In this sample, the four-week weekday mean was the stronger traditional baseline in every group. These are pilot estimates conditional on the sampled series and current model settings. The bootstrap interval does not capture uncertainty from drawing a different 5,000-series sample, model selection, or hyperparameter tuning. The bootstrap-positive share is descriptive and is not a p-value.

## Load the generated summaries

Run `python src/forecast_experiments.py --max-series 5000 --bootstrap-replicates 1000` from the repository root first. The generated data stays local and is excluded from version control.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "forecast_experiments.py").exists())
ARTIFACTS = ROOT / "artifacts"
manifest = json.loads((ARTIFACTS / "run_manifest.json").read_text(encoding="utf-8"))
fold_metrics = pd.read_csv(ARTIFACTS / "metrics.csv")
pooled_metrics = pd.read_csv(ARTIFACTS / "pooled_metrics.csv")
bootstrap = pd.read_csv(ARTIFACTS / "bootstrap_comparison.csv")
print({k: manifest[k] for k in ["seed", "series_evaluated", "horizon_days", "folds", "bootstrap_replicates", "sales_file_sha256"]})

In [ ]:
overall = pooled_metrics[pooled_metrics["volatility_group"] == "all"]
display(overall[["model", "pooled_WAPE", "pooled_MAE", "mean_fold_WAPE", "mean_fold_MAE"]].sort_values("pooled_WAPE").round(4))

group_scores = pooled_metrics.pivot(index="volatility_group", columns="model", values="pooled_WAPE")
display(group_scores.round(4))

In [ ]:
display(bootstrap[["volatility_group", "baseline_model", "pooled_baseline_WAPE", "pooled_lightgbm_WAPE", "WAPE_improvement", "improvement_95CI_low", "improvement_95CI_high", "relative_improvement_pct", "bootstrap_probability_improvement_positive", "series"]].round(4))

## Interpretation and next research step

The pilot supports a larger evaluation, not a final claim. Repeat on the full M5 population after checking available memory, then test whether adding prices and promotions improves the same chronological folds. Keep the existing baselines, fold dates, and volatility definitions fixed for that comparison. M5 results should not be generalized directly to all retail businesses.